# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and Python data analysis tools.

### Dataset Source

The dataset is described by a Croissant JSON-LD schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in your environment.
!pip install mlcroissant

## 1. Data Loading

Load metadata and available records from the dataset using `mlcroissant`. We begin by loading the metadata and inspecting some basic properties.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Explore available record sets, fields, and their unique `@id` references. All subsequent data access will use these `@id` values, consistent with Croissant requirements.

In [ ]:
# List all available record sets and their IDs

print('Available Record Sets (@id, name):')
for rs in metadata.record_sets:
    print(f"  - {rs['@id']}: {rs['name'] if 'name' in rs else ''}")

# For this dataset, let's get the fields for each record set
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

for rs in metadata.record_sets:
    print(f"\nFields in Record Set {rs['@id']}:")
    for field in rs['fields']:
        fname = field.get('name', '')
        print(f"   - {field['@id']} (name: {fname})")

## 3. Data Extraction

Extract records from each record set using its `@id` and load into a pandas DataFrame. We'll demonstrate by iterating over all available record sets.

In [ ]:
# Create a dictionary of DataFrames for each record set (@id as key)

dataframes = {}

for record_set in record_set_ids:
    print(f"Loading records from record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}\n")
    else:
        print(f"No records found for record set {record_set}.\n")

# Example: Display the head of the first loaded DataFrame
if len(dataframes) > 0:
    example_rs = next(iter(dataframes))
    print(f"Head of DataFrame for record set {example_rs}:")
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate basic EDA using a numeric field and a group field from the main record set. We will:

- Filter records above a threshold value
- Normalize the numeric column
- Group by a categorical field to show averages

**Reminder:** All field and column references are by their `@id`.

In [ ]:
# Select the main record set for analysis (choose the first one with records by default)
main_record_set = None
for k, v in dataframes.items():
    if not v.empty:
        main_record_set = k
        break

if main_record_set is None:
    print('No data available for EDA.')
else:
    df = dataframes[main_record_set]

    # Display a sample of column names (using field @id)
    print(f"Columns in {main_record_set}:")
    print(df.columns.tolist())

    # --- Example: Use a numeric field and a group field ---
    # Try to auto-select a likely numeric field and a groupable field
    import numpy as np
    numeric_field_id = None
    group_field_id = None
    
    for col in df.columns:
        # Guess numeric field if dtype is number and no obvious ID/str pattern
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    # Guess groupable field: typically low unique count and dtype object/str
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() > 1 and df[col].nunique() < len(df) * 0.5:
            group_field_id = col
            if group_field_id != numeric_field_id:
                break

    if numeric_field_id is None:
        print('No numeric field detected for EDA.')
    else:
        print(f"Using numeric field: {numeric_field_id}")

        # Filter: e.g. records above the mean
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group and aggregate
        if group_field_id is not None and group_field_id in filtered_df.columns:
            print(f"\nGrouping by {group_field_id}:")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            display(grouped.head())
        else:
            print("No suitable group field detected.")

## 5. Visualization

Visualize data distributions and relationships. For example, histogram of the numeric field and a boxplot grouped by a category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and perform basic EDA on the FAIR^2 CRC survivors dataset using the Croissant metadata and records interface. All record sets, fields, and columns were referenced via their stable `@id`. With these tools, you can explore and prepare the data for statistical analysis or machine learning according to FAIR principles.